In [ ]:
# ============================================================
# City of Austin eCheckbook - Load & Scope Dataset
# Source: City of Austin Open Data (data.austintexas.gov)
# Link: https://data.austintexas.gov/Budget-and-Finance/Austin-Finance-Online-eCheckbook/8c6z-qnmj
# ============================================================

import pandas as pd

file_path = r"C:\Users\shidesh\OneDrive - amazon.com\Desktop\HR Docs\MSDS\MSDS 430\Module 4\EDA proposal\Austin_Finance_Online_eCheckbook.csv"

# --- First, peek at the structure without loading the whole thing ---
preview = pd.read_csv(file_path, nrows=5)
print("Columns:", preview.columns.tolist())
print(f"\nPreview shape: {preview.shape}")
preview.head()

In [ ]:
# --- Load only FY2020-2025 to keep it manageable ---
# Read in chunks to avoid memory issues with the 664MB file

chunks = []
for chunk in pd.read_csv(file_path, chunksize=100_000, low_memory=False):
    filtered = chunk[chunk['FY_DC'].astype(str).isin(['2020','2021','2022','2023','2024','2025'])]
    chunks.append(filtered)
    
df = pd.concat(chunks, ignore_index=True)
print(f"Rows loaded (FY2020-2025): {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# --- Convert key fields ---
df['AMOUNT'] = pd.to_numeric(df['AMOUNT'], errors='coerce')
df['CHK_EFT_ISS_DT'] = pd.to_datetime(df['CHK_EFT_ISS_DT'], errors='coerce')
df['PER_CD'] = pd.to_numeric(df['PER_CD'], errors='coerce')
df['OBJ_CD'] = df['OBJ_CD'].astype(str)
df['COMM_CD'] = df['COMM_CD'].astype(str)

print(f"Date range: {df['CHK_EFT_ISS_DT'].min()} to {df['CHK_EFT_ISS_DT'].max()}")
print(f"Total spend: ${df['AMOUNT'].sum():,.2f}")
print(f"Unique vendors: {df['LGL_NM'].nunique():,}")
print(f"Unique departments: {df['DEPT_NM'].nunique()}")
print(f"Fiscal years: {sorted(df['FY_DC'].unique())}")

In [ ]:
# --- Step 5: info(), head(), tail() ---
df.info()

In [ ]:
df.head(3)

In [ ]:
# --- Optional: Save the scoped-down version for faster reloading ---
scoped_path = r"C:\Users\shidesh\OneDrive - amazon.com\Desktop\HR Docs\MSDS\MSDS 430\Module 4\EDA proposal\austin_echeckbook_fy2020_2025.csv"
df.to_csv(scoped_path, index=False)
print(f"Saved scoped dataset: {df.shape[0]:,} rows ({df.memory_usage(deep=True).sum() / 1024**2:.1f} MB)")